### Ce notebook permet de faire une vérification de la qualité des données avant toute consolidation, visualisation ou modélisation.

In [2]:
import pandas as pd
from pathlib import Path

###  Chargement de données

In [3]:
# Chemin d'accès aux données brutes

RAW_DATA_DIR = Path("../..") / "data" / "raw"

# Charger les datasets

files = {
    "idmc": "data_idmc_depuis_2000.csv",
    "solutions": "data_solutions_depuis_2000.csv",
    "decisions": "decisions_asile_depuis_2000.csv",
    "demandes": "demandes_asile_depuis_2000.csv",
    "demographie": "demographie_depuis_2000.csv",
    "pays": "countries.csv"
}


dfs = {
    name: pd.read_csv(RAW_DATA_DIR / filename)
    for name, filename in files.items()
}

for name, df in dfs.items():
    print(f"{name:15} : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

idmc            : 934 lignes × 10 colonnes
solutions       : 21,258 lignes × 13 colonnes
decisions       : 113,929 lignes × 17 colonnes
demandes        : 120,597 lignes × 14 colonnes
demographie     : 116,781 lignes × 24 colonnes
pays            : 232 lignes × 16 colonnes


### Cohérence géographique de l’IDMC

In [4]:
df = dfs["idmc"].copy()

coo = df["coo_iso"].astype("string").str.strip().str.upper()
coa = df["coa_iso"].astype("string").str.strip().str.upper()

same_country = coo == coa

print("Même origine/destination :", same_country.sum())
print("Différents :", (~same_country).sum())

Même origine/destination : 934
Différents : 0


In [5]:
display(
    df.loc[
        ~same_country,
        [
            "year",
            "coo_id",
            "coo_name",
            "coo_iso",
            "coa_id",
            "coa_name",
            "coa_iso",
            "total"
        ]
    ].head(50)
)

,year,coo_id,coo_name,coo_iso,coa_id,coa_name,coa_iso,total


### Cohérence des décisions d’asile

In [6]:
df = dfs["decisions"].copy()

cols = [
    "dec_recognized",
    "dec_other",
    "dec_rejected",
    "dec_closed",
    "dec_total"
]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["dec_sum"] = (
    df[
        [
            "dec_recognized",
            "dec_other",
            "dec_rejected",
            "dec_closed"
        ]
    ]
    .sum(axis=1, min_count=1)
)

df["ecart"] = df["dec_total"] - df["dec_sum"]

display(
    df[df["ecart"].fillna(0) != 0].head()
)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,procedure_type,dec_level,dec_pc,dec_recognized,dec_other,dec_rejected,dec_closed,dec_total,dec_sum,ecart
73,2000,10,Armenia,ARM,ARM,49,Czechia,CZE,CZE,G,AR,C,15,0,21,5,43,41,2
77,2000,30,Bulgaria,BUL,BGR,49,Czechia,CZE,CZE,G,AR,C,5,0,75,10,89,90,-1
78,2000,40,Congo,COB,COG,49,Czechia,CZE,CZE,G,AR,C,0,0,5,10,14,15,-1
100,2000,200,Ukraine,UKR,UKR,49,Czechia,CZE,CZE,G,AR,C,5,0,39,19,64,63,1
201,2001,262,Unknown,UKN,UNK,132,Malta,MTA,MLT,G,AR,C,0,0,0,0,5,0,5


Cette étape servera à créer **Recognition Rate** . *Recognition Rate = dec_recognized / dec_total*

### Cohérence démographique

In [9]:
df = dfs["demographie"].copy()

for col in ["f_total", "m_total", "total"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["total_calcule"] = (
    df["f_total"] + df["m_total"]
)

df["ecart"] = (
    df["total"] - df["total_calcule"]
)

print(
    "Lignes avec écart :",
    (df["ecart"].fillna(0) != 0).sum()
)

Lignes avec écart : 66185
